# 30年分年度HSIC敏感性分析（按气候区分组）

本notebook对1991-2020年每一年的数据，分别在4个气候区（HW、HD、CW、CD）中进行HSIC敏感性分析。

**分析设置：**
- 时间范围：1991-2020（30年）
- 气候区：HW（热湿）、HD（热干）、CW（冷湿）、CD（冷干）
- 敏感性方法：HSIC
- 输入变量：precipitation, lai
- 输出变量：evapotrans, tran, evspsblveg, evspsblsoi

## 0. 导入库和配置

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 设置绘图样式
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("库导入成功！")

库导入成功！


## 1. 读取数据和气候区标签

In [2]:
# 读取已打好气候区标签的30年数据（包含time列）
classic_path = Path('data/preprocessed/preprocessed_with_zones/classic_with_climate_zones_30y.csv')
lpj_path = Path('data/preprocessed/preprocessed_with_zones/lpj_guess_with_climate_zones_30y.csv')

print("读取已打好气候区标签的30年数据...")
classic_df = pd.read_csv(classic_path)
lpj_df = pd.read_csv(lpj_path)

print(f"\nCLASSIC数据: {len(classic_df):,} 行")
print(f"LPJ-GUESS数据: {len(lpj_df):,} 行")

print("\n时间范围:")
print(f"  CLASSIC: {classic_df['time'].min()} - {classic_df['time'].max()}")
print(f"  LPJ-GUESS: {lpj_df['time'].min()} - {lpj_df['time'].max()}")

print("\n气候区分布:")
print("CLASSIC:")
print(classic_df['climate_zone'].value_counts())
print("\nLPJ-GUESS:")
print(lpj_df['climate_zone'].value_counts())

读取已打好气候区标签的30年数据...

CLASSIC数据: 1,824,800 行
LPJ-GUESS数据: 1,751,623 行

时间范围:
  CLASSIC: 1991 - 2020
  LPJ-GUESS: 1991 - 2020

气候区分布:
CLASSIC:
climate_zone
CW         671688
CD         399802
HW         397473
HD         354820
Unknown      1017
Name: count, dtype: int64

LPJ-GUESS:
climate_zone
CW         671248
HW         398518
HD         356292
CD         321060
Unknown      4505
Name: count, dtype: int64


## 2. 定义HSIC敏感性分析函数

In [11]:
def rbf_kernel_separate(X, Y, sigma_x=None, sigma_y=None):
    """
    分别计算X和Y的RBF核矩阵（使用各自的带宽参数）
    
    参数:
        X: 第一个变量 (n_samples, 1)
        Y: 第二个变量 (n_samples, 1)
        sigma_x: X的RBF核带宽参数
        sigma_y: Y的RBF核带宽参数
    
    返回:
        K_X, K_Y: 两个核矩阵
    """
    # 计算X的核矩阵
    X_sq = np.sum(X**2, axis=1).reshape(-1, 1)
    dist_X_sq = X_sq + X_sq.T - 2 * X @ X.T
    
    if sigma_x is None:
        sigma_x = np.median(np.sqrt(dist_X_sq[dist_X_sq > 0]))
        if sigma_x == 0:
            sigma_x = 1.0
    
    K_X = np.exp(-dist_X_sq / (2 * sigma_x**2))
    
    # 计算Y的核矩阵
    Y_sq = np.sum(Y**2, axis=1).reshape(-1, 1)
    dist_Y_sq = Y_sq + Y_sq.T - 2 * Y @ Y.T
    
    if sigma_y is None:
        sigma_y = np.median(np.sqrt(dist_Y_sq[dist_Y_sq > 0]))
        if sigma_y == 0:
            sigma_y = 1.0
    
    K_Y = np.exp(-dist_Y_sq / (2 * sigma_y**2))
    
    return K_X, K_Y


def center_kernel_matrix(K):
    """
    中心化核矩阵
    
    参数:
        K: 核矩阵 (n, n)
    
    返回:
        中心化后的核矩阵
    """
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    return H @ K @ H


def calculate_hsic_index(df, input_var, output_var, subsample=None):
    """
    计算归一化的HSIC (Hilbert-Schmidt Independence Criterion) 敏感性指数
    
    HSIC衡量两个变量之间的独立性，值越大表示依赖性越强
    归一化后的HSIC值在[0, 1]范围内
    
    参数:
        df: 数据框
        input_var: 输入变量名
        output_var: 输出变量名
        subsample: 子采样数量（如果数据太大）
    
    返回:
        float: 归一化的HSIC指数 (0-1范围)
    """
    # 提取数据
    X = df[input_var].values
    Y = df[output_var].values
    
    # 移除NaN
    mask = ~(np.isnan(X) | np.isnan(Y))
    X = X[mask]
    Y = Y[mask]
    
    if len(X) < 10:
        return 0.0
    
    # 子采样（如果数据量太大）
    if subsample is not None and len(X) > subsample:
        indices = np.random.choice(len(X), subsample, replace=False)
        X = X[indices]
        Y = Y[indices]
    
    # 转换为列向量
    X = X.reshape(-1, 1)
    Y = Y.reshape(-1, 1)
    n = len(X)
    
    # 分别计算X和Y的核矩阵（使用各自的带宽参数）
    K_X, K_Y = rbf_kernel_separate(X, Y, sigma_x=None, sigma_y=None)
    
    # 中心化核矩阵
    K_X_centered = center_kernel_matrix(K_X)
    K_Y_centered = center_kernel_matrix(K_Y)
    
    # 计算HSIC
    hsic = np.trace(K_X_centered @ K_Y_centered) / (n - 1)**2
    
    # 归一化HSIC（除以各自的核矩阵范数）
    norm_X = np.sqrt(np.trace(K_X_centered @ K_X_centered) / (n - 1)**2)
    norm_Y = np.sqrt(np.trace(K_Y_centered @ K_Y_centered) / (n - 1)**2)
    
    if norm_X > 0 and norm_Y > 0:
        hsic_normalized = hsic / (norm_X * norm_Y)
    else:
        hsic_normalized = 0.0
    
    return hsic_normalized


print("✓ HSIC函数定义完成（归一化版本）")
print("  - 使用独立的带宽参数（X和Y各自计算）")
print("  - HSIC值归一化到[0, 1]范围")
print("  - 值越大表示依赖性越强")

✓ HSIC函数定义完成（归一化版本）
  - 使用独立的带宽参数（X和Y各自计算）
  - HSIC值归一化到[0, 1]范围
  - 值越大表示依赖性越强


## 3. 执行30年分年度HSIC分析

In [13]:
# 定义变量
input_vars = ['precipitation', 'lai']
output_vars = ['evapotrans', 'tran', 'evspsblveg', 'evspsblsoi']
climate_zones = ['HW', 'HD', 'CW', 'CD']
years = sorted(classic_df['time'].unique())

# HSIC子采样参数（如果数据量过大）
HSIC_SUBSAMPLE = 5000

print(f"分析配置:")
print(f"  年份数: {len(years)} ({years[0]}-{years[-1]})")
print(f"  气候区: {climate_zones}")
print(f"  输入变量: {input_vars}")
print(f"  输出变量: {output_vars}")
print(f"  HSIC子采样: {HSIC_SUBSAMPLE}")
print(f"  总分析数: {len(years)} × 4气候区 × 2输入 × 4输出 × 2模型 = {len(years)*4*2*4*2}")

分析配置:
  年份数: 30 (1991-2020)
  气候区: ['HW', 'HD', 'CW', 'CD']
  输入变量: ['precipitation', 'lai']
  输出变量: ['evapotrans', 'tran', 'evspsblveg', 'evspsblsoi']
  HSIC子采样: 5000
  总分析数: 30 × 4气候区 × 2输入 × 4输出 × 2模型 = 1920


In [15]:
# ============================================================================
# CLASSIC模型：HSIC分析
# ============================================================================
print("="*80)
print("CLASSIC模型：30年分年度HSIC分析")
print("="*80)

classic_hsic_results = []

for year in tqdm(years, desc="HSIC - 年份进度"):
    # 筛选该年的数据
    year_data = classic_df[classic_df['time'] == year].copy()
    
    for zone in climate_zones:
        # 筛选该气候区的数据
        zone_data = year_data[year_data['climate_zone'] == zone].copy()
        
        if len(zone_data) < 50:  # 样本数太少则跳过
            continue
        
        for output_var in output_vars:
            for input_var in input_vars:
                # 计算HSIC指数
                hsic_result = calculate_hsic_index(
                    zone_data, 
                    input_var, 
                    output_var,
                    subsample=HSIC_SUBSAMPLE if len(zone_data) > HSIC_SUBSAMPLE else None
                )
                
                # 保存结果
                classic_hsic_results.append({
                    'model': 'CLASSIC',
                    'year': year,
                    'climate_zone': zone,
                    'input_var': input_var,
                    'output_var': output_var,
                    'hsic': hsic_result,
                    'n_samples': len(zone_data)
                })

classic_hsic_df = pd.DataFrame(classic_hsic_results)
print(f"\n✓ CLASSIC HSIC分析完成，共{len(classic_hsic_df):,}条结果")
print(f"\n前10行:")
print(classic_hsic_df.head(10))

# 保存HSIC结果
output_dir = Path('output/30y_HSIC')
output_dir.mkdir(parents=True, exist_ok=True)
classic_hsic_df.to_csv(output_dir / 'classic_30y_hsic_results.csv', index=False)
print(f"\n✓ HSIC结果已保存到: {output_dir / 'classic_30y_hsic_results.csv'}")

CLASSIC模型：30年分年度HSIC分析


HSIC - 年份进度: 100%|██████████| 30/30 [59:42<00:00, 119.41s/it]


✓ CLASSIC HSIC分析完成，共960条结果

前10行:
     model  year climate_zone      input_var  output_var      hsic  n_samples
0  CLASSIC  1991           HW  precipitation  evapotrans  0.350628      13251
1  CLASSIC  1991           HW            lai  evapotrans  0.529314      13251
2  CLASSIC  1991           HW  precipitation        tran  0.234147      13251
3  CLASSIC  1991           HW            lai        tran  0.582144      13251
4  CLASSIC  1991           HW  precipitation  evspsblveg  0.465188      13251
5  CLASSIC  1991           HW            lai  evspsblveg  0.689910      13251
6  CLASSIC  1991           HW  precipitation  evspsblsoi  0.171386      13251
7  CLASSIC  1991           HW            lai  evspsblsoi  0.523000      13251
8  CLASSIC  1991           HD  precipitation  evapotrans  0.678781      11829
9  CLASSIC  1991           HD            lai  evapotrans  0.615459      11829

✓ HSIC结果已保存到: output\30y_HSIC\classic_30y_hsic_results.csv


In [16]:
# ============================================================================
# LPJ-GUESS模型：HSIC分析
# ============================================================================
print("="*80)
print("LPJ-GUESS模型：30年分年度HSIC分析")
print("="*80)

lpj_hsic_results = []

for year in tqdm(years, desc="HSIC - 年份进度"):
    # 筛选该年的数据
    year_data = lpj_df[lpj_df['time'] == year].copy()
    
    for zone in climate_zones:
        # 筛选该气候区的数据
        zone_data = year_data[year_data['climate_zone'] == zone].copy()
        
        if len(zone_data) < 50:  # 样本数太少则跳过
            continue
        
        for output_var in output_vars:
            for input_var in input_vars:
                # 计算HSIC指数
                hsic_result = calculate_hsic_index(
                    zone_data, 
                    input_var, 
                    output_var,
                    subsample=HSIC_SUBSAMPLE if len(zone_data) > HSIC_SUBSAMPLE else None
                )
                
                # 保存结果
                lpj_hsic_results.append({
                    'model': 'LPJ-GUESS',
                    'year': year,
                    'climate_zone': zone,
                    'input_var': input_var,
                    'output_var': output_var,
                    'hsic': hsic_result,
                    'n_samples': len(zone_data)
                })

lpj_hsic_df = pd.DataFrame(lpj_hsic_results)
print(f"\n✓ LPJ-GUESS HSIC分析完成，共{len(lpj_hsic_df):,}条结果")
print(f"\n前10行:")
print(lpj_hsic_df.head(10))

# 保存HSIC结果
lpj_hsic_df.to_csv(output_dir / 'lpj_30y_hsic_results.csv', index=False)
print(f"\n✓ HSIC结果已保存到: {output_dir / 'lpj_30y_hsic_results.csv'}")

LPJ-GUESS模型：30年分年度HSIC分析


HSIC - 年份进度: 100%|██████████| 30/30 [59:47<00:00, 119.58s/it] 


✓ LPJ-GUESS HSIC分析完成，共960条结果

前10行:
       model  year climate_zone      input_var  output_var      hsic  \
0  LPJ-GUESS  1991           HW  precipitation  evapotrans  0.427346   
1  LPJ-GUESS  1991           HW            lai  evapotrans  0.694324   
2  LPJ-GUESS  1991           HW  precipitation        tran  0.277708   
3  LPJ-GUESS  1991           HW            lai        tran  0.742586   
4  LPJ-GUESS  1991           HW  precipitation  evspsblveg  0.481258   
5  LPJ-GUESS  1991           HW            lai  evspsblveg  0.712719   
6  LPJ-GUESS  1991           HW  precipitation  evspsblsoi  0.057499   
7  LPJ-GUESS  1991           HW            lai  evspsblsoi  0.045759   
8  LPJ-GUESS  1991           HD  precipitation  evapotrans  0.674691   
9  LPJ-GUESS  1991           HD            lai  evapotrans  0.792303   

   n_samples  
0      13298  
1      13298  
2      13298  
3      13298  
4      13298  
5      13298  
6      13298  
7      13298  
8      11879  
9      11879  

✓ HS

In [19]:
# 合并两个模型的结果
all_hsic_df = pd.concat([classic_hsic_df, lpj_hsic_df], ignore_index=True)

print(f"合并结果统计:")
print(f"  总记录数: {len(all_hsic_df):,}")
print(f"  CLASSIC: {len(classic_hsic_df):,}")
print(f"  LPJ-GUESS: {len(lpj_hsic_df):,}")

# 导出合并结果
all_hsic_df.to_csv(output_dir / 'all_30y_hsic_results.csv', index=False)
print(f"\n✓ 合并结果已导出: {output_dir / 'all_30y_hsic_results.csv'}")

合并结果统计:
  总记录数: 1,920
  CLASSIC: 960
  LPJ-GUESS: 960

✓ 合并结果已导出: output\30y_HSIC\all_30y_hsic_results.csv


## 4. 可视化：HSIC指数时间序列图

In [20]:
def plot_hsic_timeseries_by_zone(df, model_name, output_var, metric='hsic'):
    """
    绘制HSIC指数时间序列图（按气候区分组）
    
    每个子图代表一个气候区，显示2个输入变量的时间趋势
    """
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    axes = axes.flatten()
    
    zone_colors = {
        'HW': '#FF6B6B',
        'HD': '#FFD93D',
        'CW': '#6BCB77',
        'CD': '#4D96FF'
    }
    
    zone_names = {
        'HW': 'Hot-Wet',
        'HD': 'Hot-Dry',
        'CW': 'Cold-Wet',
        'CD': 'Cold-Dry'
    }
    
    input_colors = {
        'precipitation': '#3498db',
        'lai': '#e74c3c'
    }
    
    for i, zone in enumerate(climate_zones):
        ax = axes[i]
        
        # 筛选该气候区和输出变量的数据
        zone_data = df[(df['climate_zone'] == zone) & 
                      (df['output_var'] == output_var)]
        
        if len(zone_data) == 0:
            ax.text(0.5, 0.5, f'No data for {zone}',
                   ha='center', va='center', fontsize=14)
            ax.axis('off')
            continue
        
        # 绘制每个输入变量的时间序列
        for input_var in input_vars:
            input_data = zone_data[zone_data['input_var'] == input_var].sort_values('year')
            
            if len(input_data) > 0:
                ax.plot(input_data['year'], input_data[metric],
                       marker='s', linewidth=2, markersize=4,
                       color=input_colors[input_var],
                       label=input_var.upper(),
                       alpha=0.8)
        
        ax.set_xlabel('Year', fontsize=11, fontweight='bold')
        ax.set_ylabel('HSIC Index', fontsize=11, fontweight='bold')
        ax.set_title(f'{zone} - {zone_names[zone]}',
                    fontsize=13, fontweight='bold',
                    color=zone_colors[zone])
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=10)
        
        # 设置x轴刻度（每5年）
        years_range = range(int(input_data['year'].min()), 
                          int(input_data['year'].max()) + 1, 5)
        ax.set_xticks(years_range)
        ax.tick_params(axis='x', rotation=45)
    
    fig.suptitle(f'{model_name} - HSIC Time Series for {output_var.upper()}',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    return fig

print("✓ 时间序列绘图函数定义完成")

✓ 时间序列绘图函数定义完成


In [21]:
# 为每个输出变量绘制CLASSIC模型的时间序列图
print("生成CLASSIC模型HSIC时间序列图...")

for output_var in output_vars:
    fig = plot_hsic_timeseries_by_zone(
        classic_hsic_df,
        'CLASSIC',
        output_var,
        metric='hsic'
    )
    
    # 保存图片
    fig_path = output_dir / f'classic_hsic_timeseries_{output_var}.png'
    fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ {fig_path.name}")

print("\n✓ CLASSIC模型时间序列图生成完成")

生成CLASSIC模型HSIC时间序列图...
  ✓ classic_hsic_timeseries_evapotrans.png
  ✓ classic_hsic_timeseries_tran.png
  ✓ classic_hsic_timeseries_evspsblveg.png
  ✓ classic_hsic_timeseries_evspsblsoi.png

✓ CLASSIC模型时间序列图生成完成


In [22]:
# 为每个输出变量绘制LPJ-GUESS模型的时间序列图
print("生成LPJ-GUESS模型HSIC时间序列图...")

for output_var in output_vars:
    fig = plot_hsic_timeseries_by_zone(
        lpj_hsic_df,
        'LPJ-GUESS',
        output_var,
        metric='hsic'
    )
    
    # 保存图片
    fig_path = output_dir / f'lpj_hsic_timeseries_{output_var}.png'
    fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ {fig_path.name}")

print("\n✓ LPJ-GUESS模型时间序列图生成完成")

生成LPJ-GUESS模型HSIC时间序列图...
  ✓ lpj_hsic_timeseries_evapotrans.png
  ✓ lpj_hsic_timeseries_tran.png
  ✓ lpj_hsic_timeseries_evspsblveg.png
  ✓ lpj_hsic_timeseries_evspsblsoi.png

✓ LPJ-GUESS模型时间序列图生成完成


## 5. 可视化：HSIC指数热图

In [ ]:
def plot_hsic_heatmap(df, model_name, input_var, output_var, metric='hsic'):
    """
    绘制HSIC指数热图（年份 × 气候区）
    
    参数:
        df: 结果数据框
        model_name: 模型名称
        input_var: 输入变量
        output_var: 输出变量
        metric: 指标（hsic）
    """
    # 筛选数据
    subset = df[(df['input_var'] == input_var) & 
               (df['output_var'] == output_var)]
    
    # 透视表：年份 × 气候区
    pivot_data = subset.pivot(index='year', 
                             columns='climate_zone', 
                             values=metric)
    
    # 按气候区顺序排列
    pivot_data = pivot_data[climate_zones]
    
    # 绘制热图
    fig, ax = plt.subplots(figsize=(10, 12))
    
    sns.heatmap(pivot_data, 
                cmap='viridis',
                annot=False,
                fmt='.4f',
                cbar_kws={'label': 'HSIC Index'},
                linewidths=0.5,
                ax=ax)
    
    ax.set_xlabel('Climate Zone', fontsize=12, fontweight='bold')
    ax.set_ylabel('Year', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name} - HSIC Heatmap\n'
                f'Input: {input_var.upper()}, Output: {output_var.upper()}',
                fontsize=14, fontweight='bold', pad=15)
    
    plt.tight_layout()
    return fig

print("✓ 热图绘制函数定义完成")

In [ ]:
# 为每个输入-输出组合绘制热图（CLASSIC模型）
print("生成CLASSIC模型HSIC热图...")

for input_var in input_vars:
    for output_var in output_vars:
        fig = plot_hsic_heatmap(
            classic_hsic_df,
            'CLASSIC',
            input_var,
            output_var,
            metric='hsic'
        )
        
        # 保存图片
        fig_path = output_dir / f'classic_hsic_heatmap_{input_var}_{output_var}.png'
        fig.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"  ✓ {fig_path.name}")

print("\n✓ CLASSIC模型热图生成完成")

In [ ]:
# 为每个输入-输出组合绘制热图（LPJ-GUESS模型）
print("生成LPJ-GUESS模型HSIC热图...")

for input_var in input_vars:
    for output_var in output_vars:
        fig = plot_hsic_heatmap(
            lpj_hsic_df,
            'LPJ-GUESS',
            input_var,
            output_var,
            metric='hsic'
        )
        
        # 保存图片
        fig_path = output_dir / f'lpj_hsic_heatmap_{input_var}_{output_var}.png'
        fig.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"  ✓ {fig_path.name}")

print("\n✓ LPJ-GUESS模型热图生成完成")

## 6. 统计汇总

In [ ]:
# 计算30年平均HSIC指数（按气候区和变量组合）
summary_stats = all_hsic_df.groupby(
    ['model', 'climate_zone', 'input_var', 'output_var']
).agg({
    'hsic': ['mean', 'std', 'min', 'max'],
    'n_samples': 'mean'
}).round(6)

# 展平多级列名
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]
summary_stats = summary_stats.reset_index()

print("30年平均HSIC指数统计汇总:")
print(summary_stats.head(20))

# 导出统计汇总
summary_stats.to_csv(output_dir / '30y_hsic_summary_statistics.csv', index=False)
print(f"\n✓ 统计汇总已导出: {output_dir / '30y_hsic_summary_statistics.csv'}")

## 7. 任务完成总结

In [ ]:
print("="*80)
print("30年分年度HSIC敏感性分析完成！")
print("="*80)

print("\n分析概况:")
print(f"  时间范围: {years[0]}-{years[-1]} ({len(years)}年)")
print(f"  气候区: {', '.join(climate_zones)}")
print(f"  模型: CLASSIC, LPJ-GUESS")
print(f"  输入变量: {', '.join(input_vars)}")
print(f"  输出变量: {', '.join(output_vars)}")
print(f"  HSIC子采样: {HSIC_SUBSAMPLE}")

print("\n结果统计:")
print(f"  CLASSIC结果: {len(classic_hsic_df):,} 条")
print(f"  LPJ-GUESS结果: {len(lpj_hsic_df):,} 条")
print(f"  总计: {len(all_hsic_df):,} 条")

print("\n输出文件:")
print(f"  目录: {output_dir}")
print("  CSV文件:")
print("    - classic_30y_hsic_results.csv")
print("    - lpj_30y_hsic_results.csv")
print("    - all_30y_hsic_results.csv")
print("    - 30y_hsic_summary_statistics.csv")

print("\n可视化图表:")
print(f"  HSIC时间序列图: {len(output_vars)} × 2模型 = {len(output_vars)*2} 张")
print(f"  HSIC热图: {len(input_vars)} × {len(output_vars)} × 2模型 = {len(input_vars)*len(output_vars)*2} 张")

print("\n✓ 分析完成！")